In [ ]:
!pip install gensim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from nltk.tokenize import word_tokenize
import nltk
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import gensim.downloader as api

In [ ]:
data_1 = pd.read_csv("/content/drive/MyDrive/ML/ham-spam-dataset/spam.csv", encoding='ISO-8859-1')

In [ ]:
data_1

In [ ]:
data_2 = pd.read_csv("/content/drive/MyDrive/ML/ham-spam-dataset/spam_dataset.csv")

In [ ]:
data_2

In [ ]:
data_3 = pd.read_csv("/content/drive/MyDrive/ML/ham-spam-dataset/spam_ham_dataset.csv")


In [ ]:
data_3

In [ ]:
data_1.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

In [ ]:
data_1.columns = ["label", "text"]

In [ ]:
data_1["label"] = (data_1['label'] == "spam").astype("int8")

In [ ]:
data_1.label.value_counts()

In [ ]:
data_2.columns = ["text", "label"]

In [ ]:
data_2.label.value_counts()

In [ ]:
data_3.drop(columns=['Unnamed: 0', 'label'], inplace = True)

In [ ]:
data_3.columns = ["text", "label"]

In [ ]:
data = pd.concat([data_1, data_2, data_3], axis=0).reset_index(drop=True)

In [ ]:
data.label.value_counts()

In [ ]:
x = data.text
y = data.label.values

In [ ]:
y

In [ ]:
wordpunct_tokenize(x[0].lower())

In [ ]:
def preprocess(text):
  text = "".join([i for i in text.lower() if i.isalpha() or i == " " or i == "'"])
  lst = " ".join(word_tokenize(text))
  return lst

In [ ]:
preprocess(x[10])

In [ ]:
X = []
for t in tqdm(x):
  X.append(preprocess(t)

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
tf_idf = TfidfVectorizer(min_df=3)
x_train_tfidf = tf_idf.fit_transform(x_train)
x_test_tfidf = tf_idf.transform(x_test)

In [ ]:
x_train_tfidf

In [ ]:
x_test_tfidf

In [ ]:
model = LogisticRegression(class_weight='balanced')
model.fit(x_train_tfidf, y_train)
train_pred = model.predict(x_train_tfidf)
test_pred = model.predict(x_test_tfidf)
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)
print("Train accuracy = ", train_acc)
print("Train f1 score = ", train_f1)
print("Test accuracy = ", test_acc)
print("Test f1 score = ", test_f1)

In [ ]:
def predict(text):
  return model.predict(tf_idf.transform([preprocess(text)]))

In [ ]:
t = """your text here"""

predict(t)

In [ ]:
model_w2v = api.load("word2vec-google-news-300")

In [ ]:
def get_sentence_embedding(tokens, modelw2v = model_w2v):
  valid_vectors = [model_w2v[word] for word in tokens if word in model_w2v]
  if not valid_vectors:
    return np.zeros(model_w2v.vector_size)
  return np.mean(valid_vectors, axis=0)

In [ ]:
def get_embeddings(sentences, modelw2v = model_w2v):
 embs = np.zeros((len(sentences), modelw2v.vector_size))
 for i, sent in enumerate(tqdm(sentences)):
   embs[i] = get_sentence_embedding(sent.split())
 return embs

In [ ]:
x_train_embs = get_embeddings(x_train)
x_test_embs = get_embeddings(x_test)

In [ ]:
x_train_embs.shape, x_test_embs.shape

In [ ]:
model = LogisticRegression(class_weight='balanced')
model.fit(x_train_embs, y_train)
train_pred = model.predict(x_train_embs)
test_pred = model.predict(x_test_embs)
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)
print("Train accuracy = ", train_acc)
print("Train f1 score = ", train_f1)
print("Test accuracy = ", test_acc)
print("Test f1 score = ", test_f1)